In [0]:
# Inject WHL to bypass Serverless environment dependencies crash


# cell 0 — 安装 WHL（Serverless ）

# **1. Dependencies + Configurations**

In [0]:
import json
import time
import uuid
from datetime import datetime, timezone, timedelta
from typing import List, Dict, Any, Optional
import requests

from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

# ---------------------------------------------------------
# Configuration & Widgets
# ---------------------------------------------------------
dbutils.widgets.text("catalog", "enterprise")
dbutils.widgets.text("schema", "bronze_market")
dbutils.widgets.text("table", "crypto_ohlc_raw")
dbutils.widgets.text("source", "coinbase")
dbutils.widgets.text("symbols", "BTCUSDT,ETHUSDT")
dbutils.widgets.text("interval", "1d")
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("sleep_ms", "250")
dbutils.widgets.text("max_days", "7")
dbutils.widgets.text("repair_start_date", "")
dbutils.widgets.text("repair_end_date", "")

#######
dbutils.widgets.text("mode", "realtime")  # backfill | realtime
dbutils.widgets.text("safety_lag_minutes", "2")  # avoid incomplete candles
dbutils.widgets.text("state_schema", "gold_observability")
dbutils.widgets.text("state_table", "ingestion_state")
dbutils.widgets.text("default_lookback_minutes", "120")  # used if no state yet
######

# Constants
CATALOG    = dbutils.widgets.get("catalog").strip()
SCHEMA     = dbutils.widgets.get("schema").strip()
TABLE      = dbutils.widgets.get("table").strip()
TARGET_TBL = f"{CATALOG}.{SCHEMA}.{TABLE}"
SOURCE     = dbutils.widgets.get("source").strip()
SYMBOLS    = [s.strip().upper() for s in dbutils.widgets.get("symbols").split(",") if s.strip()]
INTERVAL   = dbutils.widgets.get("interval").strip()
START_DATE = dbutils.widgets.get("start_date").strip()
END_DATE   = dbutils.widgets.get("end_date").strip()
SLEEP_MS   = int(dbutils.widgets.get("sleep_ms"))
MAX_DAYS   = int(dbutils.widgets.get("max_days"))
REPAIR_START_DATE = dbutils.widgets.get("repair_start_date").strip()
REPAIR_END_DATE   = dbutils.widgets.get("repair_end_date").strip()

#######
MODE = dbutils.widgets.get("mode").strip().lower()
SAFETY_LAG_MIN = int(dbutils.widgets.get("safety_lag_minutes"))
STATE_SCHEMA = dbutils.widgets.get("state_schema").strip()
STATE_TABLE  = dbutils.widgets.get("state_table").strip()
DEFAULT_LOOKBACK_MIN = int(dbutils.widgets.get("default_lookback_minutes"))
STATE_TBL = f"{CATALOG}.{STATE_SCHEMA}.{STATE_TABLE}"
if MODE not in ("backfill","realtime"):
    raise ValueError("mode must be backfill or realtime")
#####

In [0]:
print(f"[DEBUG] widget interval = {INTERVAL}")
print(f"[DEBUG] start_date={START_DATE}, end_date={END_DATE}, mode={MODE}, target={TARGET_TBL}")


# **2. Time Utility Helpers**

In [0]:
%skip
def utc_now() -> datetime:
    return datetime.now(timezone.utc)

def iso_utc(ts: datetime) -> str:
    return ts.isoformat(timespec="seconds").replace("+00:00", "Z")

def parse_yyyy_mm_dd(s: str) -> datetime:
    return datetime.strptime(s, "%Y-%m-%d").replace(tzinfo=timezone.utc)

def to_ms(dt: datetime) -> int:
    """UTC datetime -> epoch milliseconds"""
    return int(dt.timestamp() * 1000)

def ms_to_ts(ms: int) -> datetime:
    return datetime.fromtimestamp(ms / 1000.0, tz=timezone.utc)

In [0]:
def parse_yyyy_mm_dd(s: str) -> datetime:
    return datetime.strptime(s, "%Y-%m-%d").replace(tzinfo=timezone.utc)

def utc_now():
    return datetime.now(timezone.utc)

def normalize_date_str(s: str | None) -> str | None:
    if s is None:
        return None
    s = str(s).strip()
    return s if s else None

START_DATE = normalize_date_str(START_DATE)
END_DATE   = normalize_date_str(END_DATE)
REPAIR_START_DATE = normalize_date_str(REPAIR_START_DATE)
REPAIR_END_DATE   = normalize_date_str(REPAIR_END_DATE)

# 支持 end_date=today
if END_DATE is not None and END_DATE.lower() == "today":
    END_DATE = utc_now().strftime("%Y-%m-%d")
if REPAIR_END_DATE is not None and REPAIR_END_DATE.lower() == "today":
    REPAIR_END_DATE = utc_now().strftime("%Y-%m-%d")

# end_date 为空 -> 今天
if END_DATE is None:
    END_DATE = utc_now().strftime("%Y-%m-%d")

# start_date 为空 -> 当天（只跑今天）
if START_DATE is None:
    START_DATE = END_DATE

start_dt = parse_yyyy_mm_dd(START_DATE)
end_dt   = parse_yyyy_mm_dd(END_DATE)

if end_dt < start_dt:
    raise ValueError(f"END_DATE {END_DATE} must be >= START_DATE {START_DATE}")

num_days = (end_dt.date() - start_dt.date()).days + 1
if num_days > MAX_DAYS:
    raise ValueError(f"Requested {num_days} days exceeds limit of {MAX_DAYS}.")

days = [start_dt + timedelta(days=i) for i in range(num_days)]
RUN_ID = str(uuid.uuid4())
INGESTION_TS = utc_now()


# **3. Exchange API Client**

In [0]:
CB_BASE = "https://api.exchange.coinbase.com"
PROVIDER = "coinbase"   # 强制只走 Coinbase，不允许 fallback

def iso_utc_z(dt: datetime) -> str:
    """Coinbase 需要 RFC3339/Z 格式"""
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")

def fetch_klines(symbol: str, interval: str, start_dt: datetime, end_dt: datetime):
    if PROVIDER != "coinbase":
        raise RuntimeError(f"Provider disabled: {PROVIDER}")
    return fetch_klines_coinbase_range(symbol, interval, start_dt, end_dt)

def fetch_klines_coinbase_range(symbol: str, interval: str, start_dt: datetime, end_dt: datetime) -> Dict[str, Any]:
    product_id = symbol.replace("USDT", "-USD")
    if interval == "1d":
        granularity = 86400
    elif interval == "1h":
        granularity = 3600
    elif interval == "1m":
        granularity = 60
    else:
        raise ValueError("Coinbase: only 1m / 1h / 1d supported")


    url = f"{CB_BASE}/products/{product_id}/candles"
    params = {
        "start": iso_utc_z(start_dt),
        "end": iso_utc_z(end_dt),
        "granularity": granularity
    }
    headers = {"User-Agent": "Mozilla/5.0", "Accept": "application/json"}

    max_retries = 6
    base_sleep = 1.0  # seconds

    last_err = None

    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=30)

            print("COINBASE FINAL URL:", r.url)
            print("COINBASE status:", r.status_code)
            print("COINBASE server:", r.headers.get("server"))
            print("COINBASE body head:", r.text[:200])

            # 成功
            if r.status_code == 200:
                kl = r.json()
                return {
                    "symbol": symbol,
                    "interval": interval,
                    "start_utc": params["start"],
                    "end_utc": params["end"],
                    "kline_count": len(kl),
                    "klines": kl
                }

            # 需要重试的状态码：429 / 5xx
            if r.status_code == 429 or (500 <= r.status_code < 600):
                last_err = f"{r.status_code}: {r.text[:200]}"
            else:
                # 其他错误（例如 400/401/403）直接失败，不要重试
                raise RuntimeError(f"Coinbase API Error {r.status_code}: {r.text[:200]}")

        except Exception as e:
            # 网络异常等也允许重试
            last_err = str(e)

        # ---- backoff ----
        if attempt < max_retries:
            # 指数退避 + 一点 jitter，避免同时撞到服务端抖动
            sleep_s = base_sleep * (2 ** (attempt - 1)) + (0.1 * attempt)
            print(f"[WARN] Coinbase retry {attempt}/{max_retries} after {sleep_s:.1f}s, last_err={last_err}")
            time.sleep(sleep_s)
        else:
            break

    raise RuntimeError(f"Coinbase API failed after retries. symbol={symbol}, start={params['start']}, end={params['end']}, last_err={last_err}")

def backfill_klines_one_day(symbol: str, interval: str, day_start: datetime) -> List[Dict[str, Any]]:
    """
    - 1m: 300分钟一段（<=300根）
    - 1h: 一天一段（24根）
    - 1d: 一天一根，直接一段（1根）
    """
    if day_start.tzinfo is None:
        day_start = day_start.replace(tzinfo=timezone.utc)

    day_end = day_start + timedelta(days=1)

    if interval == "1m":
        step = timedelta(minutes=300)
    elif interval == "1h":
        step = timedelta(hours=24)
    elif interval == "1d":
        step = timedelta(days=1)
    else:
        raise ValueError("Coinbase backfill: only 1m / 1h / 1d supported")

    pages: List[Dict[str, Any]] = []
    cur = day_start
    while cur < day_end:
        nxt = min(cur + step, day_end)
        page = fetch_klines(symbol, interval, cur, nxt)
        pages.append(page)
        cur = nxt

        if SLEEP_MS > 0:
            time.sleep(SLEEP_MS / 1000.0)

    return pages

# **4. Ingestion Window Logic**

In [0]:
# Resolve extraction window
end_dt = parse_yyyy_mm_dd(END_DATE) if END_DATE else utc_now().replace(hour=0, minute=0, second=0, microsecond=0)

if MODE == "backfill":
    if not START_DATE:
        raise ValueError("backfill mode requires start_date (YYYY-MM-DD)")

    start_dt = parse_yyyy_mm_dd(START_DATE)
    if end_dt < start_dt:
        raise ValueError(f"END_DATE {END_DATE} must be >= START_DATE {START_DATE}")

    num_days = (end_dt.date() - start_dt.date()).days + 1
    if num_days > MAX_DAYS:
        raise ValueError(f"Requested {num_days} days exceeds limit of {MAX_DAYS}.")

    shared_days = [start_dt + timedelta(days=i) for i in range(num_days)]
    days_by_symbol = {sym: shared_days for sym in SYMBOLS}

else:
    wm_by_symbol = {}
    if spark.catalog.tableExists(TARGET_TBL):
        wm_rows = (
            spark.table(TARGET_TBL)
            .where((F.col("source") == SOURCE) & (F.col("interval") == INTERVAL))
            .groupBy("symbol")
            .agg(F.max("event_time").alias("mx"))
            .collect()
        )
        wm_by_symbol = {r["symbol"].upper(): r["mx"] for r in wm_rows if r["mx"] is not None}

    lookback_days = max(1, (DEFAULT_LOOKBACK_MIN + 1439) // 1440)
    days_by_symbol = {}

    for sym in SYMBOLS:
        if START_DATE:
            sym_start = parse_yyyy_mm_dd(START_DATE)
        else:
            wm = wm_by_symbol.get(sym)
            if wm is not None:
                sym_start = wm.astimezone(timezone.utc).replace(
                    hour=0, minute=0, second=0, microsecond=0
                ) + timedelta(days=1)
            else:
                sym_start = (end_dt - timedelta(days=lookback_days)).replace(
                    hour=0, minute=0, second=0, microsecond=0
                )

        if sym_start > end_dt:
            days_by_symbol[sym] = []
            continue

        req_days = (end_dt.date() - sym_start.date()).days + 1
        if req_days > MAX_DAYS:
            raise ValueError(f"{sym}: requested {req_days} days exceeds limit of {MAX_DAYS}.")

        days_by_symbol[sym] = [sym_start + timedelta(days=i) for i in range(req_days)]

if REPAIR_START_DATE:
    repair_start_dt = parse_yyyy_mm_dd(REPAIR_START_DATE)
    repair_end_token = REPAIR_END_DATE or END_DATE
    repair_end_dt = parse_yyyy_mm_dd(repair_end_token)

    if repair_end_dt < repair_start_dt:
        raise ValueError(f"repair_end_date {repair_end_token} must be >= repair_start_date {REPAIR_START_DATE}")

    repair_days = [
        repair_start_dt + timedelta(days=i)
        for i in range((repair_end_dt.date() - repair_start_dt.date()).days + 1)
    ]

    for sym in SYMBOLS:
        merged = sorted(set(days_by_symbol.get(sym, []) + repair_days))
        if len(merged) > MAX_DAYS:
            raise ValueError(
                f"{sym}: requested {len(merged)} days exceeds limit of {MAX_DAYS}. "
                "Increase max_days for repair run."
            )
        days_by_symbol[sym] = merged

    print(
        f"[INFO] repair window injected: {REPAIR_START_DATE} -> {repair_end_token} "
        f"({len(repair_days)} days per symbol)"
    )

RUN_ID = str(uuid.uuid4())
INGESTION_TS = utc_now()

total_days = sum(len(v) for v in days_by_symbol.values())
print(f"[INFO] mode={MODE}, total symbol-day tasks={total_days}")
for sym in SYMBOLS:
    sym_days = days_by_symbol.get(sym, [])
    if sym_days:
        print(f"[INFO] {sym}: {sym_days[0].date()} -> {sym_days[-1].date()} ({len(sym_days)} days)")
    else:
        print(f"[INFO] {sym}: up-to-date, no fetch")

if total_days == 0:
    dbutils.notebook.exit("SKIPPED_NO_NEW_CRYPTO_DAYS")


# **5. Target Schema & DDL**

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TARGET_TBL} (
  source STRING,
  symbol STRING,
  interval STRING,
  event_time TIMESTAMP,
  raw_json STRING,
  run_id STRING,
  ingestion_ts TIMESTAMP
)
USING DELTA
""")

# **6. Ingestion Execution**

In [0]:
import sys
from pathlib import Path

_local_src = str(Path.cwd() / "src")
if _local_src not in sys.path:
    sys.path.insert(0, _local_src)

from lakehouse.ingestion import fetch_pages_concurrently

rows: List[Row] = []
errors: List[Dict[str, Any]] = []
all_pages = []
kept_pages = 0
skipped_empty_pages = 0

for sym in SYMBOLS:
    sym_days = days_by_symbol.get(sym, [])
    if not sym_days:
        continue

    print(f"[INFO] Fetching {sym} for {len(sym_days)} day(s)...")
    sym_pages, sym_errors = fetch_pages_concurrently(
        symbols=[sym],
        days=sym_days,
        interval=INTERVAL,
        fetch_func=backfill_klines_one_day,
        max_workers=4,
    )
    all_pages.extend(sym_pages)
    errors.extend(sym_errors)

for sym, day_start, pages in all_pages:
    for p in pages:
        kline_count = int(p.get("kline_count", 0) or 0)
        if kline_count <= 0:
            skipped_empty_pages += 1
            continue

        kept_pages += 1
        rows.append(Row(
            source=PROVIDER,
            symbol=sym,
            interval=INTERVAL,
            event_time=day_start.replace(tzinfo=timezone.utc),
            raw_json=json.dumps(p, separators=(",", ":")),
            run_id=RUN_ID,
            ingestion_ts=INGESTION_TS
        ))

print(
    f"[INFO] bronze pages kept={kept_pages}, skipped_empty={skipped_empty_pages}, "
    f"rows_to_write={len(rows)}"
)

if errors:
    print(f"[WARN] Failures occurred: {len(errors)}")
    for x in errors[:50]:
        print(x)

if rows:
    schema = StructType([
        StructField("source", StringType(), False),
        StructField("symbol", StringType(), False),
        StructField("interval", StringType(), False),
        StructField("event_time", TimestampType(), True),
        StructField("raw_json", StringType(), False),
        StructField("run_id", StringType(), False),
        StructField("ingestion_ts", TimestampType(), False),
    ])
    df = spark.createDataFrame(rows, schema=schema).dropDuplicates()
    df.write.mode("append").format("delta").saveAsTable(TARGET_TBL)
